## RFM Calculation

### Objective
Compute RFM (Recency, Frequency, Monetary) metrics for each customer based on the processed Online Sales dataset.

- **Recency (R):** Number of days since the customer's most recent transaction relative to a reference date.
- **Frequency (F):** Total number of transactions made by the customer.
- **Monetary (M):** Total amount spent by the customer, calculated using the transaction value after outlier handling.

### Input Data
The input dataset is `Online_Sales_with_VIP.csv`, which includes transaction-level data with computed transaction values and customer segmentation.

### Methodology

1. Convert `Transaction_Date` to datetime format.
2. Define a snapshot date as one day after the latest transaction date.
3. Group data by `CustomerID`.
4. Compute:
   - Recency: difference between snapshot date and the most recent transaction
   - Frequency: number of unique transactions
   - Monetary: total transaction value

### Output Data

Two versions of the RFM dataset are generated:

- **`customer_profile_rfm.csv`**: essential for exploratory data analysis, feature engineering, and transformation decisions. In particular, it allows for proper assessment of data distribution and supports meaningful business interpretation.

- **`customer_profile_rfm_scaled.csv`**: designed for clustering algorithms, which rely on distance-based calculations. Without scaling, features with larger magnitudes (such as Monetary) would dominate the clustering process and lead to biased segmentation.
- A selective log transformation is applied to highly skewed features (Frequency and Monetary) before scaling to reduce distribution imbalance.

In [1]:
import pandas as pd
from IPython.display import display

sales_clean = pd.read_csv(
    "../data/processed/Online_Sales_with_VIP.csv",
    parse_dates=["Transaction_Date"]
)

snapshot_date = sales_clean["Transaction_Date"].max() + pd.Timedelta(days=1)

rfm_df = (
    sales_clean
    .groupby("CustomerID")
    .agg(
        Recency=("Transaction_Date", lambda x: (snapshot_date - x.max()).days),
        Frequency=("Transaction_ID", "nunique"),
        Monetary=("Transaction_Value", "sum")
    )
    .reset_index()
)

display(rfm_df.head(2))

rfm_df.to_csv(
    "../data/processed/customer_profile_rfm.csv",
    index=False
)

,CustomerID,Recency,Frequency,Monetary
0,12346,108,1,180.99
1,12347,60,31,14500.04


In [2]:
cols = ["Recency", "Frequency", "Monetary"]

skew_df = pd.DataFrame({
    "Feature": cols,
    "Skewness": [rfm_df[col].skew() for col in cols]
})

display(skew_df.sort_values("Skewness", ascending=False))

,Feature,Skewness
2,Monetary,6.747279
1,Frequency,5.689494
0,Recency,0.454355


### Skewness-Based Transformation Decision

The skewness analysis indicates that Monetary and Frequency are highly right-skewed, while Recency remains relatively symmetric.

Monetary (skewness = 6.75) and Frequency (skewness = 5.69) exhibit extreme positive skewness, reflecting the presence of a small number of customers with disproportionately high transaction values or purchase frequencies. To mitigate the impact of these extreme values and stabilize the distribution, a log transformation is applied to both features.

In contrast, Recency shows low skewness (0.45), indicating an approximately symmetric distribution. Therefore, no transformation is applied in order to preserve its original scale and interpretability.

This selective transformation ensures that only features with significant distribution imbalance are adjusted, while well-behaved features remain unchanged.

In [3]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# save scale version
rfm_features = rfm_df[["Recency", "Frequency", "Monetary"]].copy()
rfm_features["Frequency"] = np.log1p(rfm_features["Frequency"])
rfm_features["Monetary"] = np.log1p(rfm_features["Monetary"])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

rfm_ready = pd.DataFrame(
    rfm_scaled,
    columns=["Recency", "Frequency", "Monetary"]
)

rfm_ready.insert(0, "CustomerID", rfm_df["CustomerID"])

display(rfm_ready.head())

rfm_ready.to_csv(
    "../data/processed/customer_profile_rfm_scaled.csv",
    index=False
)

,CustomerID,Recency,Frequency,Monetary
0,12346,-0.365961,-1.782571,-1.451163
1,12347,-0.837001,1.012911,1.486702
2,12348,-0.699614,-0.266074,0.024247
3,12350,-1.249160,0.023983,-0.040704
4,12356,-0.365961,0.179406,0.183935
